# Crawling Data detik.com — Sport & Finance

Tugas: mengumpulkan **200 data berita** dari detik.com
- 100 data pertama: kategori **sport** (label `sport`)
- 100 data berikutnya: kategori **finance** (label `finance`)

## Kolom data wajib
| Kolom | Isi |
|-------|-----|
| `id` | nomor urut integer 1–200 (1–100 = sport, 101–200 = finance) |
| `isi_berita` | teks utama artikel hasil ekstraksi *trafilatura* |
| `label` | kategori berita (`sport` atau `finance`) |

---

**Etika & robots.txt**: detik.com mengizinkan crawl (`User-agent: * Allow: /`),
dengan larangan tertentu (mis. `*/indeks/`, `*&sortby`). Notebook ini memanfaatkan
**sitemap resmi detik** untuk mengumpulkan URL artikel dan memberi jeda antar-request.
Gunakan hanya untuk kepentingan pembelajaran/tugas kuliah.

> **Catatan hasil:** Hasil crawl sudah berhasil dibuat dan tersimpan di file
> `crawling_detik.csv` dan `crawling_detik.json`. Bagian paling bawah notebook
> **memuat hasil dari file** agar tidak perlu meng-crawl ulang (hemat & aman
> dari koneksi yang tidak stabil). Kode untuk *crawl ulang* tetap disertakan
> sebagai referensi pada bagian tengah notebook.

In [1]:
# ============================================================
# SEL 1 — Setup: import & konfigurasi dasar
# ============================================================
import re
import time
import json
import requests
import pandas as pd
from pathlib import Path
import trafilatura

# Header User-Agent agar permintaan tidak diblokir sebagai bot dasar
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0 Safari/537.36"
}

# Politeness delay (detik) antar-request agar tidak membebani server
POLITE_DELAY = 1.0

print("Setup OK — trafilatura", trafilatura.__version__)

Setup OK — trafilatura 2.2.0


---
## Bagian A — Kode crawling (referensi)

Fungsi-fungsi di bawah adalah kode yang **sudah dijalankan sekali** untuk
menghasilkan file `crawling_detik.csv` / `.json`. Kamu **tidak wajib**
menjalankannya lagi (bisa langkah besar / mengirim ~200 request). Cukup
baca dan pahami; hasil akhir dimuat dari file di Bagian B.
---

In [2]:
# ============================================================
# SEL 2 — Helper: kumpulkan URL artikel dari sitemap detik
# (REFERENSI — tidak dijalankan agar tidak meng-crawl ulang)
# ============================================================

def get_urls_from_sitemap(sitemap_url):
    """Ambil seluruh URL artikel dari sebuah sitemap XML."""
    r = requests.get(sitemap_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    # URL artikel dibungkus <![CDATA[...]]>
    return re.findall(r"<loc>\s*<!\[CDATA\[\s*([^\]]+?)\s*\]\]>\s*</loc>", r.text)


def collect_article_urls(kanal, limit=100):
    """Kumpulkan <limit> URL artikel unik dari sitemap kanal detik."""
    base = f"https://{kanal}.detik.com"
    idx = requests.get(f"{base}/sitemap.xml", headers=HEADERS, timeout=30)
    idx.raise_for_status()
    news_sitemaps = re.findall(r"https?://[^\s<\[]+?sitemap_news\.xml", idx.text)

    urls = set()
    for sm in news_sitemaps:
        try:
            for u in get_urls_from_sitemap(sm):
                urls.add(u)
        except Exception as e:
            print(f"  [skip] {sm} -> {e}")
        if len(urls) >= limit:
            break
        time.sleep(POLITE_DELAY)
    return list(urls)[:limit]


print("Helper sitemap siap (definisi fungsi saja, belum dijalankan).")

Helper sitemap siap (definisi fungsi saja, belum dijalankan).


In [3]:
# ============================================================
# SEL 3 — Helper: crawl isi berita (REFERENSI)
# ============================================================

def crawl_article(url, retry=3, backoff=(1, 2, 4)):
    """Ambil teks utama artikel memakai trafilatura, dengan retry ringan."""
    for attempt in range(retry):
        try:
            downloaded = trafilatura.fetch_url(url)
            if downloaded:
                text = trafilatura.extract(downloaded, include_comments=False)
                if text:
                    return text
        except Exception:
            pass
        if attempt < retry - 1:
            time.sleep(backoff[attempt])
    return None


def crawl_many(urls, label):
    results = []
    for i, url in enumerate(urls, start=1):
        isi = crawl_article(url)
        results.append({"url": url, "isi_berita": isi, "label": label})
        if i % 10 == 0 or i == len(urls):
            print(f"  [{label}] {i}/{len(urls)} selesai")
        time.sleep(POLITE_DELAY)
    return results


print("Helper crawl siap (definisi fungsi saja, belum dijalankan).")

Helper crawl siap (definisi fungsi saja, belum dijalankan).


---
## Bagian B — Memuat hasil crawl dari file

Baris-baris berikut **membaca hasil yang sudah tersimpan** (`crawling_detik.csv`
dan `crawling_detik.json`) dan menampilkannya. Tidak ada request jaringan.
---

In [4]:
# ============================================================
# SEL 4 — Muat hasil dari file (tanpa crawl ulang)
# ============================================================
CSV_PATH = Path("crawling_detik.csv")
JSON_PATH = Path("crawling_detik.json")

if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    print(f"Memuat {len(df)} baris dari {CSV_PATH.name}")
else:
    # Fallback: jika file belum ada, buat dari JSON atau kosongkan
    if JSON_PATH.exists():
        df = pd.read_json(JSON_PATH)
        print(f"Memuat {len(df)} baris dari {JSON_PATH.name}")
    else:
        df = pd.DataFrame(columns=["id", "isi_berita", "label", "url"])
        print("File hasil belum ditemukan — biarkan kosong.")

print("Kolom:", list(df.columns))
df.head()

Memuat 200 baris dari crawling_detik.csv
Kolom: ['id', 'isi_berita', 'label', 'url']


,id,isi_berita,label,url
0,1,Misi Indonesia untuk meloloskan tim 3x3 ke Oli...,sport,https://sport.detik.com/basket/d-8608204/perba...
1,2,Inilah daftar pelatih NBA terbaik sepanjang ma...,sport,https://sport.detik.com/basket/d-8296454/10-pe...
2,3,"Pelita Jaya mendapat dukungan ""tenaga"" baru un...",sport,https://sport.detik.com/basket/d-8353921/agar-...
3,4,Center Miami Heat Bam Adebayo mencetak 83 poin...,sport,https://sport.detik.com/basket/d-8395308/heat-...
4,5,Campus League 2026 Basketball Jakarta sudah me...,sport,https://sport.detik.com/basket/d-8512605/campu...


In [5]:
# ============================================================
# SEL 5 — Pemeriksaan data: kolom wajib & distribusi label
# ============================================================
print("Ukuran DataFrame:", df.shape)
print("\nJumlah per label:\n", df["label"].value_counts().to_string())
print("\nMissing isi_berita:", df["isi_berita"].isna().sum())
print("Head (5 data pertama = sport):")
print(df.head(5)[["id", "label", "url"]].to_string(index=False))
print("\nTail (5 data terakhir = finance):")
print(df.tail(5)[["id", "label", "url"]].to_string(index=False))
print("\nid urut 1..200?", list(df["id"]) == list(range(1, len(df) + 1)))

Ukuran DataFrame: (200, 4)

Jumlah per label:
 label
sport      100
finance    100

Missing isi_berita: 0
Head (5 data pertama = sport):
 id label                                                                                                         url
  1 sport   https://sport.detik.com/basket/d-8608204/perbasi-gandeng-pakar-fiba-untuk-kembangkan-basket-3x3-indonesia
  2 sport       https://sport.detik.com/basket/d-8296454/10-pelatih-dengan-kemenangan-terbanyak-sepanjang-sejarah-nba
  3 sport                            https://sport.detik.com/basket/d-8353921/agar-pelita-jaya-makin-besar-di-jakarta
  4 sport  https://sport.detik.com/basket/d-8395308/heat-tekuk-wizards-adebayo-lewati-rekor-poin-per-laga-kobe-bryant
  5 sport https://sport.detik.com/basket/d-8512605/campus-league-2026-serie-jakarta-tuntas-lanjut-ke-tingkat-nasional

Tail (5 data terakhir = finance):
 id   label                                                                                                             

In [6]:
# ============================================================
# SEL 6 — Contoh: tampilkan isi berita salah satu artikel
# ============================================================
row = df[df["label"] == "sport"].iloc[0]
print(f"id={row['id']} | label={row['label']}")
print("url :", row["url"])
print("\nisi_berita (awal):\n", row["isi_berita"][:300])

id=1 | label=sport
url : https://sport.detik.com/basket/d-8608204/perbasi-gandeng-pakar-fiba-untuk-kembangkan-basket-3x3-indonesia

isi_berita (awal):
 Misi Indonesia untuk meloloskan tim 3x3 ke Olimpiade mulai digencarkan. PP Perbasi menggandeng Nicolas Widmer, salah satu agensi 3x3 yang dipercaya FIBA.
Kerja sama itu diawali dengan menggelar workshop selama dua hari (4-5/8), di Kantor Perbasi, GBK Arena Senayan. Dengan tujuan, memetakan jalan Ind


---
## Bagian C — (Opsional) Crawl ulang dari awal

Bila kamu *ingin* meng-crawl ulang 200 artikel (butuh internet stabil & beberapa
menit, mengirim ~200 request), jalankan sel-sel di Bagian C. Hasil akan
menimpa file `crawling_detik.csv` / `.json` yang sudah ada.
---

In [7]:
# ============================================================
# SEL 7 (OPSIONAL) — Crawl ulang sport & finance
# Peringatan: mengirim request ke detik; butuh internet stabil.
# ============================================================
N = 100

_re_crawl = False
if _re_crawl:
    print("Mengumpulkan URL sport & finance...")
    sport_urls = collect_article_urls("sport", N)
    finance_urls = collect_article_urls("finance", N)

    print("Crawl sport...")
    sport_rows = crawl_many(sport_urls, "sport")
    print("Crawl finance...")
    finance_rows = crawl_many(finance_urls, "finance")

    all_rows = sport_rows + finance_rows
    df = pd.DataFrame(all_rows)
    df.insert(0, "id", range(1, len(df) + 1))
    df = df[["id", "isi_berita", "label", "url"]]
    df = df.dropna(subset=["isi_berita"]).reset_index(drop=True)

    df.to_csv("crawling_detik.csv", index=False, encoding="utf-8")
    df.to_json("crawling_detik.json", orient="records", force_ascii=False, indent=2)
    print(f"Crawl ulang selesai: {len(df)} baris. File ditimpa.")
else:
    print("Re-crawl dinonaktifkan. Ubah `_re_crawl = True` bila ingin crawl ulang.")

Re-crawl dinonaktifkan. Ubah `_re_crawl = True` bila ingin crawl ulang.
